In [2]:
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding
import numpy as np

#### Q1: embedding a query with fastembed

In [67]:
model_handle = "jinaai/jina-embeddings-v2-small-en"

q = 'I just discovered the course. Can I join now?'

embedding_model = TextEmbedding(model_name=model_handle)
query_embedding = embedding_model.embed(q)

query_embedding_list = list(query_embedding)[0]


#print(list(query_embedding))
print(np.min(query_embedding_list))
print(len(query_embedding_list))


-0.11726373551188797
512


#### Q2: cosine similarity with another vector

In [68]:
doc = 'Can I still join the course after the start date?'

doc_embedding = embedding_model.embed(doc)
doc_embedding_list = list(doc_embedding)[0]


query_doc_similarity = np.array(query_embedding_list).dot(np.array(doc_embedding_list))

print(f"the cosine similarity between query and doc is: {query_doc_similarity}")

the cosine similarity between query and doc is: 0.9008528856818037


#### Q3: ranking by cosine

In [71]:
documents = [
    {
        'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
        'section': 'General course-related questions',
        'question': 'Course - Can I still join the course after the start date?',
        'course': 'data-engineering-zoomcamp'
        }, 
    {
        'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
        'section': 'General course-related questions',
        'question': 'Course - Can I follow the course after it finishes?',
        'course': 'data-engineering-zoomcamp'
        },
    {
        'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
        'section': 'General course-related questions',
        'question': 'Course - When will the course start?',
        'course': 'data-engineering-zoomcamp'
        },
    {   'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (installed with Anaconda)\nTerraform\nGit\nLook over the prerequisites and syllabus to see if you are comfortable with these subjects.',
        'section': 'General course-related questions',
        'question': 'Course - What can I do before the course starts?',
        'course': 'data-engineering-zoomcamp'
        },
    {
        'text': 'Star the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.',
        'section': 'General course-related questions',
        'question': 'How can we contribute to the course?',
        'course': 'data-engineering-zoomcamp'
        }
]

In [72]:
documents_embeddings_lists = []


for document in documents: 

    text = document['text']
    text_embedding = embedding_model.embed(text)

    documents_embeddings_lists.append(list(text_embedding)[0])



documents_embeddings_matrix = np.array(documents_embeddings_lists)
query_embedding_array = np.array(query_embedding_list)

query_documents_similarities = documents_embeddings_matrix.dot(query_embedding_array)

print(f"the document index with the highest similarity is: {np.argmax(query_documents_similarities)}")


the document index with the highest similarity is: 1


#### Q4: ranking by cosine, version two

In [73]:
documents_embeddings_lists_v2 = []


for document in documents: 

    full_text = document['question'] + ' ' + document['text']
    full_embedding = embedding_model.embed(full_text)

    documents_embeddings_lists_v2.append(list(full_embedding)[0])



documents_embeddings_matrix_v2 = np.array(documents_embeddings_lists_v2)

query_documents_similarities_v2 = documents_embeddings_matrix_v2.dot(query_embedding_array)

print(f"the document index with the highest similarity is: {np.argmax(query_documents_similarities_v2)}")

the document index with the highest similarity is: 0


#### Q5: selecting the embedding model

In [74]:
textEmbedding_dims = []

for model in TextEmbedding.list_supported_models():
    textEmbedding_dims.append(model["dim"])

textEmbedding_dims = np.array(textEmbedding_dims)
print(f"the smallest dimensionality for models in fastembed is: {np.min(textEmbedding_dims)}")

the smallest dimensionality for models in fastembed is: 384


#### Q6: indexing qwith qdrant

In [75]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()


documents = []

for course in documents_raw:
    course_name = course['course']
    if course_name != 'machine-learning-zoomcamp':
        continue

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [ ]:
### selected model
model_handle_2 = 'BAAI/bge-small-en'

### setting embedding size
EMBEDDING_DIMENSIONALITY = 384

### client
client = QdrantClient("http://localhost:6333") 

### creating a collection
# Define the collection name
collection_name = "llmzoomcamp-hmw2"

# Create the collection with specified vector parameters
existing_collections = [c.name for c in client.get_collections().collections]

if collection_name not in existing_collections:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=EMBEDDING_DIMENSIONALITY,
            distance=models.Distance.COSINE
        )
    )
    print(f"Collection '{collection_name}' created.")
    
else:
    print(f"Collection '{collection_name}' already exists!.")
    client.delete_collection(collection_name=collection_name)
    print(f"Collection {collection_name} deleted, re-run to create it again.")



### create, embed and insert points into the collection
points = []
id = 0

for doc in documents:

    point = models.PointStruct(
        id=id,
        vector=models.Document(text = doc['question'] + ' ' + doc['text'], model=model_handle_2), 
        payload={
            "ful_text": doc['question'] + ' ' + doc['text'],
            "section": doc['section'],
            "course": doc['course']
        } #save all needed metadata fields
    )
    points.append(point)

    id += 1



### upserting 
client.upsert(
    collection_name=collection_name,
    points=points
)

Collection 'llmzoomcamp-hmw2' created.


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [78]:
def search(query, limit=1):

    results = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model=model_handle_2 
        ),
        limit=limit, # top closest matches
        with_payload=True #to get metadata in the results
    )

    return results



result = search(q)

print(f"Query:\n{q}\n")
print("Top Retrieved Answer:\n{}\n".format(result.points[0].payload['full_text']))
print("The score for the top retrieved answer is: {}".format(result.points[0].score))

Query:
I just discovered the course. Can I join now?

Top Retrieved Answer:
The course has already started. Can I still join it? Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.
In order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.

The score for the top retrieved answer is: 0.8703172
